In [ ]:
"""
RAG-Based Organization Chatbot System with Turkish Gemma Model
===============================================================
Uses ytu-ce-cosmos/Turkish-Gemma-9b-T1-GGUF for local inference.
Expects pre-chunked Q&A format data from your friend.
"""

# ============================================================================
# STEP 1: INSTALL REQUIRED LIBRARIES
# ============================================================================

!pip install -q langchain langchain-community
!pip install -q chromadb sentence-transformers
!pip install -q llama-cpp-python
!pip install -q huggingface-hub

print("✅ All libraries installed successfully!")

# ============================================================================
# STEP 2: IMPORT LIBRARIES
# ============================================================================

import os
import json
from typing import List, Dict
from pathlib import Path

# LangChain imports
from langchain.docstore.document import Document
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain.llms.base import LLM

# Llama CPP
from llama_cpp import Llama

print("✅ Libraries imported successfully!")

# ============================================================================
# STEP 4: CONFIGURATION
# ============================================================================

class Config:
    """Configuration for the RAG system"""

    # Embedding model (Turkish support)
    EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

    # Vector database
    VECTOR_DB_PATH = "./chroma_db"
    COLLECTION_NAME = "org_knowledge_turkish"

    # Turkish Gemma Model Configuration
    GEMMA_REPO_ID = "ytu-ce-cosmos/Turkish-Gemma-9b-T1-GGUF"
    GEMMA_FILENAME = "*Q4_K.gguf"  # Q4_K is good balance of quality/speed

    # Gemma inference parameters
    GEMMA_PARAMS = {
        "n_threads": 4,
        "n_ctx": 2048,  # Context window
        "n_predict": 512,  # Max tokens to generate
        "top_k": 20,
        "top_p": 0.95,
        "temp": 0.6,
        "repeat_penalty": 1.05,
    }

    # Retrieval settings
    TOP_K_RESULTS = 3  # Number of relevant Q&A pairs to retrieve

config = Config()

# ============================================================================
# STEP 5: DATA LOADING (Pre-chunked Q&A Format)
# ============================================================================

class QADataLoader:
    """Loads pre-chunked Q&A data from various formats"""

    @staticmethod
    def load_json_qa(file_path: str) -> List[Document]:
        """Load Q&A pairs from JSON format"""
        documents = []

        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)

            qa_pairs = data.get('data', [])

            for item in qa_pairs:
                # Combine Q&A into a single text chunk
                content = f"Soru: {item['question']}\n\nCevap: {item['answer']}"

                metadata = {
                    "question": item['question'],
                    "answer": item['answer'],
                    "category": item.get('category', 'Genel'),
                    "keywords": item.get('keywords', []),
                    "source": file_path
                }

                documents.append(Document(page_content=content, metadata=metadata))

            print(f"✅ Loaded {len(documents)} Q&A pairs from {file_path}")
            return documents

        except Exception as e:
            print(f"❌ Error loading {file_path}: {str(e)}")
            return []

    @staticmethod
    def load_text_qa(file_path: str) -> List[Document]:
        """Load Q&A pairs from text format (Q: ... A: ... separated by ---)"""
        documents = []

        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                content = f.read()

            # Split by separator
            qa_blocks = content.split('---')

            for block in qa_blocks:
                block = block.strip()
                if not block:
                    continue

                # Extract Q, A, and optional Category
                lines = block.split('\n')
                question = ""
                answer = ""
                category = "Genel"

                for line in lines:
                    line = line.strip()
                    if line.startswith('Q:'):
                        question = line[2:].strip()
                    elif line.startswith('A:'):
                        answer = line[2:].strip()
                    elif line.startswith('Category:'):
                        category = line[9:].strip()

                if question and answer:
                    content = f"Soru: {question}\n\nCevap: {answer}"
                    metadata = {
                        "question": question,
                        "answer": answer,
                        "category": category,
                        "source": file_path
                    }
                    documents.append(Document(page_content=content, metadata=metadata))

            print(f"✅ Loaded {len(documents)} Q&A pairs from {file_path}")
            return documents

        except Exception as e:
            print(f"❌ Error loading {file_path}: {str(e)}")
            return []

    @staticmethod
    def load_from_directory(dir_path: str) -> List[Document]:
        """Load all Q&A files from a directory"""
        all_docs = []

        for file_path in Path(dir_path).rglob('*'):
            if file_path.suffix.lower() == '.json':
                docs = QADataLoader.load_json_qa(str(file_path))
                all_docs.extend(docs)
            elif file_path.suffix.lower() == '.txt':
                docs = QADataLoader.load_text_qa(str(file_path))
                all_docs.extend(docs)

        print(f"✅ Total Q&A pairs loaded: {len(all_docs)}")
        return all_docs

# ============================================================================
# STEP 6: VECTOR STORE (Same as before)
# ============================================================================

class VectorStore:
    """Manages the vector database"""

    def __init__(self, embedding_model_name: str, db_path: str, collection_name: str):
        print("⏳ Initializing Turkish embedding model...")
        self.embeddings = HuggingFaceEmbeddings(
            model_name=embedding_model_name,
            model_kwargs={'device': 'cpu'}
        )
        self.db_path = db_path
        self.collection_name = collection_name
        self.vectorstore = None
        print("✅ Embedding model ready!")

    def create_from_documents(self, documents: List[Document]):
        """Create a new vector store from Q&A documents"""
        print(f"⏳ Creating vector embeddings for {len(documents)} Q&A pairs...")

        self.vectorstore = Chroma.from_documents(
            documents=documents,
            embedding=self.embeddings,
            persist_directory=self.db_path,
            collection_name=self.collection_name
        )

        print("✅ Vector store created and persisted!")
        return self.vectorstore

    def load_existing(self):
        """Load an existing vector store"""
        print("⏳ Loading existing vector store...")

        self.vectorstore = Chroma(
            persist_directory=self.db_path,
            embedding_function=self.embeddings,
            collection_name=self.collection_name
        )

        print("✅ Vector store loaded!")
        return self.vectorstore

# ============================================================================
# STEP 7: TURKISH GEMMA LLM WRAPPER
# ============================================================================

class TurkishGemmaLLM(LLM):
    """Custom LLM wrapper for Turkish Gemma model"""

    model: Llama = None

    def __init__(self, repo_id: str, filename: str, **kwargs):
        super().__init__()
        print("⏳ Downloading and loading Turkish Gemma model...")
        print("   (This may take several minutes on first run)")

        self.model = Llama.from_pretrained(
            repo_id=repo_id,
            filename=filename,
            verbose=False,
            **config.GEMMA_PARAMS
        )

        print("✅ Turkish Gemma model loaded!")

    @property
    def _llm_type(self) -> str:
        return "turkish_gemma"

    def _call(self, prompt: str, stop=None) -> str:
        """Generate response from the model"""

        # Format prompt with Gemma's special tokens
        formatted_prompt = f"<bos><start_of_turn>user\n{prompt}<end_of_turn>\n<start_of_turn>model\n"

        # Generate response
        response = self.model(
            formatted_prompt,
            stop=["<end_of_turn>", "</s>"],
            max_tokens=config.GEMMA_PARAMS['n_predict']
        )

        return response['choices'][0]['text'].strip()

# ============================================================================
# STEP 8: RAG PIPELINE
# ============================================================================

class TurkishRAGChatbot:
    """RAG chatbot using Turkish Gemma model"""

    def __init__(self, vectorstore):
        """Initialize the RAG chatbot with Turkish Gemma"""

        # Initialize Turkish Gemma LLM
        self.llm = TurkishGemmaLLM(
            repo_id=config.GEMMA_REPO_ID,
            filename=config.GEMMA_FILENAME
        )

        # Create the custom prompt template (in Turkish)
        self.prompt_template = """Sen organizasyonumuz için yardımcı bir asistansın.
Görevin, SADECE aşağıda verilen "Bağlam" kısmındaki bilgileri kullanarak soruları yanıtlamaktır.

KRİTİK KURALLAR:
1. SADECE verilen bağlamdaki bilgileri kullan
2. Eğer cevap bağlamda yoksa, şunu söyle: "Bu bilgi bilgi tabanımda bulunmuyor."
3. Genel bilgilerini veya internetten bilgileri KULLANMA
4. Yardımcı, kısa ve doğru ol
5. Bağlamdan aldığın bilgileri belirtebilirsin

Bağlam:
{context}

Kullanıcının Sorusu: {question}

Cevabın:"""

        self.PROMPT = PromptTemplate(
            template=self.prompt_template,
            input_variables=["context", "question"]
        )

        # Create the retrieval chain
        self.qa_chain = RetrievalQA.from_chain_type(
            llm=self.llm,
            chain_type="stuff",
            retriever=vectorstore.as_retriever(
                search_kwargs={"k": config.TOP_K_RESULTS}
            ),
            return_source_documents=True,
            chain_type_kwargs={"prompt": self.PROMPT}
        )

        print("✅ Turkish RAG Chatbot initialized and ready!")

    def ask(self, question: str) -> Dict:
        """Ask a question and get an answer"""
        print(f"\n❓ Soru: {question}")

        result = self.qa_chain.invoke({"query": question})

        answer = result['result']
        sources = result['source_documents']

        print(f"\n💡 Cevap: {answer}")
        print(f"\n📚 Bilgi tabanından {len(sources)} kaynak kullanıldı")

        # Show which Q&A pairs were used
        for i, doc in enumerate(sources, 1):
            print(f"\nKaynak {i}:")
            print(f"  Kategori: {doc.metadata.get('category', 'N/A')}")
            print(f"  Soru: {doc.metadata.get('question', 'N/A')[:100]}...")

        return {
            "answer": answer,
            "sources": sources,
            "num_sources": len(sources)
        }

# ============================================================================
# STEP 9: MAIN WORKFLOW
# ============================================================================

def build_turkish_rag_system(qa_data_path: str):
    """
    Build the Turkish RAG system with pre-chunked Q&A data

    Args:
        qa_data_path: Path to directory containing Q&A files (JSON or TXT)
                     or path to a single Q&A file
    """

    print("=" * 70)
    print("🚀 TURKISH RAG CHATBOT SYSTEM KURULUMU")
    print("=" * 70)

    # Step 1: Load pre-chunked Q&A data
    print("\n📂 ADIM 1: Q&A verisi yükleniyor...")

    if Path(qa_data_path).is_dir():
        documents = QADataLoader.load_from_directory(qa_data_path)
    elif qa_data_path.endswith('.json'):
        documents = QADataLoader.load_json_qa(qa_data_path)
    elif qa_data_path.endswith('.txt'):
        documents = QADataLoader.load_text_qa(qa_data_path)
    else:
        print("❌ Desteklenmeyen dosya formatı! JSON veya TXT kullanın.")
        return None

    if not documents:
        print("❌ Hiç Q&A verisi yüklenemedi!")
        return None

    # Step 2: Create vector store
    print("\n🧠 ADIM 2: Vektör veritabanı oluşturuluyor...")
    vector_store = VectorStore(
        embedding_model_name=config.EMBEDDING_MODEL,
        db_path=config.VECTOR_DB_PATH,
        collection_name=config.COLLECTION_NAME
    )
    vectorstore = vector_store.create_from_documents(documents)

    # Step 3: Initialize chatbot
    print("\n🤖 ADIM 3: Turkish Gemma RAG chatbot başlatılıyor...")
    chatbot = TurkishRAGChatbot(vectorstore)

    print("\n" + "=" * 70)
    print("✅ RAG SİSTEMİ HAZIR!")
    print("=" * 70)

    return chatbot

print("\n" + "=" * 70)
print("📖 MODÜL YÜKLENDİ - Kullanıma hazır!")
print("=" * 70)